In [2]:
import dnachisel

print(dnachisel.__version__)

3.2.16


In [4]:
import numpy
numpy.random.seed(123)

In [5]:
from dnachisel import *

# DEFINE THE OPTIMIZATION PROBLEM

problem = DnaOptimizationProblem(
    sequence=random_dna_sequence(10000),
    constraints=[
        AvoidPattern("BsaI_site"),
        EnforceGCContent(mini=0.3, maxi=0.7, window=50),
        EnforceTranslation(location=(500, 1400))
    ],
    objectives=[CodonOptimize(species='e_coli', location=(500, 1400))]
)  # Note: always use a codon optimisation specification with EnforceTranslation

# SOLVE THE CONSTRAINTS, OPTIMIZE WITH RESPECT TO THE OBJECTIVE

problem.resolve_constraints()
problem.optimize()

# PRINT SUMMARIES TO CHECK THAT CONSTRAINTS PASS

print(problem.constraints_text_summary())
print(problem.objectives_text_summary())

# GET THE FINAL SEQUENCE (AS STRING OR ANNOTATED BIOPYTHON RECORDS)

final_sequence = problem.sequence  # string
final_record = problem.to_record(with_sequence_edits=True)

location:   0%|                        | 0/2 [00:00<?, ?it/s, now=4435-4441(+)]
                                                                               
location:   0%|                           | 0/3 [00:00<?, ?it/s, now=3453-3503]
                                                                               
location:   0%|                           | 0/210 [00:00<?, ?it/s, now=500-503]
                                                                               
location:   0%|                           | 0/210 [00:00<?, ?it/s, now=500-503]
                                                                               

===> SUCCESS - all constraints evaluations pass
✔PASS ┍ AvoidPattern[0-10000](pattern:BsaI(GGTCTC))
      │ Passed. Pattern not found !
✔PASS ┍ EnforceGCContent[0-10000](mini:0.30, maxi:0.70, window:50)
      │ Passed !
✔PASS ┍ EnforceTranslation[500-1400(+)]
      │ Enforced by nucleotides restrictions


===> TOTAL OBJECTIVES SCORE:     -8.89
     -8.89 ┍ MaximizeCAI[500-1400](e_coli) 
           │ Codon opt. on window 500-1400 scored -8.89E+00




In [6]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

record = SeqRecord(
    Seq(final_sequence),
    id="optimized_sequence",
    description="DNA Chisel optimized sequence"
)

SeqIO.write(record, "optimized_sequence.fasta", "fasta")

1

In [7]:
"""
Generate 5 synthetic spike-in backbone sequences (2000 bp each), one per
target GC level (26/36/46/56/66%), using DNA Chisel.
 
Constraints applied per sequence, matching the design criteria discussed
(mirroring Zaramela et al. 2022's synDNA design rules):
  - GC content held within +/-0.5% of the target (global).
  - No self-complementary / hairpin stem >= 15 bp.
  - No exact repeated 15-mer anywhere in the sequence (i.e., no exact
    repeat of 15 bp or longer occurring more than once).
 
This script does NOT perform NCBI novelty screening (BLAST) or primer
design -- those are separate downstream steps.
"""
 
import csv
from dnachisel import (
    DnaOptimizationProblem,
    EnforceGCContent,
    AvoidHairpins,
    UniquifyAllKmers,
    random_dna_sequence,
)
 
LENGTH = 2000
GC_TARGETS = [0.26, 0.36, 0.46, 0.56, 0.66]
GC_TOLERANCE = 0.005  # +/- 0.5% GC, i.e. about +/-10 bp wiggle room on 2000 bp
SEED_BASE = 42
 
records = []
summaries = []
 
for i, gc in enumerate(GC_TARGETS):
    seed = SEED_BASE + i
    starting_seq = random_dna_sequence(LENGTH, gc_share=gc, seed=seed)
 
    problem = DnaOptimizationProblem(
        sequence=starting_seq,
        constraints=[
            EnforceGCContent(mini=gc - GC_TOLERANCE, maxi=gc + GC_TOLERANCE),
            AvoidHairpins(stem_size=15, hairpin_window=200),
            UniquifyAllKmers(15),
        ],
        logger=None,
    )
 
    problem.resolve_constraints()
 
    final_seq = problem.sequence
    actual_gc = (final_seq.count("G") + final_seq.count("C")) / LENGTH
    passed = problem.all_constraints_pass()
 
    name = f"synDNA_GC{int(round(gc * 100))}"
    records.append((name, final_seq))
    summaries.append(
        {
            "name": name,
            "target_gc_percent": round(gc * 100, 1),
            "actual_gc_percent": round(actual_gc * 100, 2),
            "length_bp": len(final_seq),
            "all_constraints_pass": passed,
        }
    )
 
    print(f"=== {name} ===")
    print(f"target GC: {gc*100:.1f}%  |  actual GC: {actual_gc*100:.2f}%  |  constraints pass: {passed}")
    print(problem.constraints_text_summary())
    print()
 
# Write FASTA (70 chars/line)
fasta_path = "synDNA_candidate_sequences.fasta"
with open(fasta_path, "w") as f:
    for name, seq in records:
        f.write(f">{name} length=2000bp\n")
        for j in range(0, len(seq), 70):
            f.write(seq[j : j + 70] + "\n")
 
# Write summary CSV
csv_path = "synDNA_candidate_summary.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["name", "target_gc_percent", "actual_gc_percent", "length_bp", "all_constraints_pass"],
    )
    writer.writeheader()
    for row in summaries:
        writer.writerow(row)
 
print(f"Wrote {fasta_path} and {csv_path}")

=== synDNA_GC26 ===
target GC: 26.0%  |  actual GC: 26.50%  |  constraints pass: True
===> SUCCESS - all constraints evaluations pass
✔PASS ┍ EnforceGCContent[0-2000](mini:0.26, maxi:0.27)
      │ Passed !
✔PASS ┍ AvoidHairpins[0-2000](stem_size:15, hairpin_window:200)
      │ Score:         0. Locations: []
✔PASS ┍ UniquifyAllKmers[0-2000(+)](k:15)
      │ Passed: no nonunique 15-mer found.



=== synDNA_GC36 ===
target GC: 36.0%  |  actual GC: 35.55%  |  constraints pass: True
===> SUCCESS - all constraints evaluations pass
✔PASS ┍ EnforceGCContent[0-2000](mini:0.35, maxi:0.36)
      │ Passed !
✔PASS ┍ AvoidHairpins[0-2000](stem_size:15, hairpin_window:200)
      │ Score:         0. Locations: []
✔PASS ┍ UniquifyAllKmers[0-2000(+)](k:15)
      │ Passed: no nonunique 15-mer found.



=== synDNA_GC46 ===
target GC: 46.0%  |  actual GC: 46.50%  |  constraints pass: True
===> SUCCESS - all constraints evaluations pass
✔PASS ┍ EnforceGCContent[0-2000](mini:0.46, maxi:0.47)
      │ Passed 

In [8]:
"""
NCBI novelty screening for candidate synDNA spike-in sequences.

Submits each sequence in a FASTA file to NCBI's remote BLASTN service and
flags any significant hit against real sequence databases, following the
same logic Zaramela et al. (2022) used for their synDNA candidates
(BLASTN, E-value cutoff < 0.01).

IMPORTANT: this script must be run somewhere with normal outbound internet
access to NCBI (blast.ncbi.nlm.nih.gov) -- e.g. your own laptop, not a
network-restricted sandbox. Requires: pip install biopython

Usage:
    python run_ncbi_novelty_screen.py synDNA_candidate_sequences.fasta

Notes on databases:
    - "nt" is NCBI's core nucleotide collection and is the right default.
      The old separate "est" and "gss" divisions the original paper also
      screened against were formally retired in 2019; their records are
      now folded into "nt" (and related divisions like "wgs"), so a
      single nt search already covers that ground.
    - For a broader net (draft/unassembled eukaryotic genomes), you can
      additionally screen against "wgs" by changing DATABASE below.

Runtime: each 2000 bp query against nt typically takes a few minutes on
NCBI's queue. With 5 queries submitted in one multi-FASTA job (the
default here), expect this to run for a while -- that's normal, not a
hang. NCBI's usage policy asks that automated clients not poll more than
once every ~10 seconds and not run many large jobs in parallel; the
Biopython qblast client already respects sensible polling behavior.
"""

import sys
import time
from Bio import SeqIO
from Bio.Blast import NCBIWWW, NCBIXML

DATABASE = "nt"
PROGRAM = "blastn"
EVALUE_CUTOFF = 0.01  # same stringency Zaramela et al. used to call a sequence "negligible identity"


def screen_fasta(fasta_path, out_report="novelty_screen_report.txt"):
    records = list(SeqIO.parse(fasta_path, "fasta"))
    if not records:
        print(f"No sequences found in {fasta_path}")
        return

    print(f"Submitting {len(records)} sequence(s) from {fasta_path} to NCBI {PROGRAM} vs {DATABASE} ...")
    print("This can take several minutes per sequence -- please be patient.")

    # Submit all records in one multi-FASTA query; qblast returns one
    # result set covering all of them.
    multi_fasta = "".join(f">{r.id}\n{str(r.seq)}\n" for r in records)

    t0 = time.time()
    result_handle = NCBIWWW.qblast(
        PROGRAM,
        DATABASE,
        multi_fasta,
        expect=10,          # retrieve a generous candidate list; we apply our own stricter cutoff below
        hitlist_size=5,      # top 5 hits per query is enough to tell if anything significant turns up
        megablast=True,
    )
    print(f"NCBI search finished in {time.time() - t0:.0f} seconds. Parsing results...")

    blast_records = NCBIXML.parse(result_handle)

    with open(out_report, "w") as report:
        for rec in blast_records:
            query_name = rec.query
            report.write(f"=== Query: {query_name} ===\n")

            significant_hits = []
            for alignment in rec.alignments:
                for hsp in alignment.hsps:
                    if hsp.expect < EVALUE_CUTOFF:
                        significant_hits.append((alignment.title, hsp.expect, hsp.identities, hsp.align_length))

            if not significant_hits:
                verdict = f"PASS -- no hit with E-value < {EVALUE_CUTOFF} found. Sequence appears novel."
                report.write(verdict + "\n\n")
                print(f"{query_name}: {verdict}")
            else:
                verdict = f"FLAG -- {len(significant_hits)} hit(s) with E-value < {EVALUE_CUTOFF}:"
                report.write(verdict + "\n")
                print(f"{query_name}: {verdict}")
                for title, evalue, identities, align_length in significant_hits:
                    pct_id = 100.0 * identities / align_length
                    line = f"    E={evalue:.2e}  identity={pct_id:.1f}%  over {align_length} bp  |  {title[:100]}"
                    report.write(line + "\n")
                    print(line)
                report.write("\n")

    print(f"\nFull report written to {out_report}")


if __name__ == "__main__":
    if len(sys.argv) != 2:
        print("Usage: python run_ncbi_novelty_screen.py <fasta_file>")
        sys.exit(1)
    screen_fasta(sys.argv[1])

Usage: python run_ncbi_novelty_screen.py <fasta_file>


SystemExit: 1

/Users/pascalhabluetzel/miniforge3/envs/eDNA_spike-in/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [10]:
screen_fasta("synDNA_candidate_sequences.fasta")

Submitting 5 sequence(s) from synDNA_candidate_sequences.fasta to NCBI blastn vs nt ...
This can take several minutes per sequence -- please be patient.
NCBI search finished in 62 seconds. Parsing results...
synDNA_GC26: PASS -- no hit with E-value < 0.01 found. Sequence appears novel.
synDNA_GC36: PASS -- no hit with E-value < 0.01 found. Sequence appears novel.
synDNA_GC46: PASS -- no hit with E-value < 0.01 found. Sequence appears novel.
synDNA_GC56: PASS -- no hit with E-value < 0.01 found. Sequence appears novel.
synDNA_GC66: PASS -- no hit with E-value < 0.01 found. Sequence appears novel.

Full report written to novelty_screen_report.txt


In [12]:
"""
Embed a universal 16S rRNA V4 primer pair (EMP protocol: 515F Parada /
806R Apprill) into each of the 5 synDNA backbone sequences, following the
"rDNA-mimic" pattern discussed: keep the primer-binding flanks as real,
conserved sequence (so universal primers actually anneal), but keep the
region between them, and the rest of the 2000 bp backbone, synthetic and
novel -- so the construct stays usable for direct shotgun coverage
quantification AND becomes compatible with 16S V4 amplicon sequencing.

Degenerate primer sequences (IUPAC codes) are resolved to one concrete,
synthesizable variant each (any one member of a degenerate primer pool
will still anneal the real mixed-base primer stock to this construct).

Primer pair (EMP 16S V4, https://earthmicrobiome.ucsd.edu/protocols-and-standards/16s/):
    515F (Parada): 5'-GTGYCAGCMGCCGCGGTAA-3'   (Y=C/T, M=A/C)
    806R (Apprill): 5'-GGACTACNVGGGTWTCTAAT-3' (N=any, V=A/C/G, W=A/T)
"""

from Bio.Seq import Seq
from dnachisel import (
    DnaOptimizationProblem,
    EnforceGCContent,
    AvoidHairpins,
    UniquifyAllKmers,
    EnforceSequence,
    random_dna_sequence,
)

LENGTH = 2000
GC_TARGETS = [0.26, 0.36, 0.46, 0.56, 0.66]
GC_TOLERANCE = 0.005
SEED_BASE = 42

# Degenerate bases resolved to one concrete variant each.
FWD_PRIMER = "GTGCCAGCAGCCGCGGTAA"     # 515F (Parada), Y->C, M->A
REV_PRIMER = "GGACTACAGGGGTTTCTAAT"    # 806R (Apprill), N->A, V->G, W->T
REV_PRIMER_RC = str(Seq(REV_PRIMER).reverse_complement())

FWD_START = 200
FWD_END = FWD_START + len(FWD_PRIMER)          # 219
INTERNAL_SPACER_LEN = 214                       # targets a ~253 bp V4-like amplicon
REV_START = FWD_END + INTERNAL_SPACER_LEN       # 433
REV_END = REV_START + len(REV_PRIMER_RC)        # 453
AMPLICON_SIZE = REV_END - FWD_START

records = []
summaries = []

for i, gc in enumerate(GC_TARGETS):
    seed = SEED_BASE + i
    starting_seq = random_dna_sequence(LENGTH, gc_share=gc, seed=seed)

    problem = DnaOptimizationProblem(
        sequence=starting_seq,
        constraints=[
            EnforceSequence(sequence=FWD_PRIMER, location=(FWD_START, FWD_END)),
            EnforceSequence(sequence=REV_PRIMER_RC, location=(REV_START, REV_END)),
            EnforceGCContent(mini=gc - GC_TOLERANCE, maxi=gc + GC_TOLERANCE),
            AvoidHairpins(stem_size=15, hairpin_window=200),
            UniquifyAllKmers(15),
        ],
        logger=None,
    )

    problem.resolve_constraints()
    final_seq = problem.sequence
    actual_gc = (final_seq.count("G") + final_seq.count("C")) / LENGTH
    passed = problem.all_constraints_pass()

    # sanity: confirm the primer sites actually landed where intended
    fwd_ok = final_seq[FWD_START:FWD_END] == FWD_PRIMER
    rev_ok = final_seq[REV_START:REV_END] == REV_PRIMER_RC

    name = f"synDNA_GC{int(round(gc * 100))}_515F806R"
    records.append((name, final_seq))
    summaries.append(
        {
            "name": name,
            "target_gc_percent": round(gc * 100, 1),
            "actual_gc_percent": round(actual_gc * 100, 2),
            "length_bp": len(final_seq),
            "fwd_primer_515F_pos": f"{FWD_START+1}-{FWD_END}",
            "rev_primer_806R_pos": f"{REV_START+1}-{REV_END}",
            "amplicon_size_bp": AMPLICON_SIZE,
            "fwd_primer_placed_correctly": fwd_ok,
            "rev_primer_placed_correctly": rev_ok,
            "all_constraints_pass": passed,
        }
    )

    print(f"=== {name} ===")
    print(f"target GC: {gc*100:.1f}%  actual GC: {actual_gc*100:.2f}%  constraints pass: {passed}")
    print(f"515F at {FWD_START+1}-{FWD_END} correct: {fwd_ok}  |  806R at {REV_START+1}-{REV_END} correct: {rev_ok}")
    print(problem.constraints_text_summary())
    print()

# FASTA output
fasta_path = "synDNA_candidate_sequences_with_primers.fasta"
with open(fasta_path, "w") as f:
    for name, seq in records:
        f.write(f">{name} length=2000bp amplicon=515F-806R({AMPLICON_SIZE}bp)\n")
        for j in range(0, len(seq), 70):
            f.write(seq[j : j + 70] + "\n")

# Summary CSV
import csv

csv_path = "synDNA_primer_embedding_summary.csv"
with open(csv_path, "w", newline="") as f:
    fieldnames = list(summaries[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in summaries:
        writer.writerow(row)

print(f"Forward primer (515F): {FWD_PRIMER}  ({len(FWD_PRIMER)} nt)")
print(f"Reverse primer (806R): {REV_PRIMER}  ({len(REV_PRIMER)} nt)")
print(f"Reverse primer binding site on (+) strand: {REV_PRIMER_RC}")
print(f"Amplicon size (515F start to 806R-site end): {AMPLICON_SIZE} bp")
print(f"Wrote {fasta_path} and {csv_path}")

# Standalone primer-primer dimer check (515F vs 806R), independent of the
# template -- this is the correct way to apply this check, since checking
# the primers against the full construct would also (correctly, but
# unhelpfully) flag their own intended binding sites as "dimers".
#
# No primer3 dependency here on purpose: primer3-py's bioconda build does
# not yet support Python 3.14 (brand new as of late 2025), so this does a
# simple, dependency-free complementarity scan instead of a full
# thermodynamic (dG/Tm) prediction. It applies the same heuristic
# discussed earlier: flag a pair if they share a run of >=4 consecutive
# complementary bases anchored at either primer's 3' end (the end that
# actually gets extended, so dimers there are the ones that matter most),
# or >=8 complementary bases in total in the best alignment.
COMPLEMENT = str.maketrans("ACGT", "TGCA")


def best_complementary_alignment(seq1, seq2):
    """Slide seq1 against the reverse of seq2 and find the alignment with
    the most complementary base pairs and the longest consecutive run."""
    seq2_rc_source = seq2.translate(COMPLEMENT)  # complement, not yet reversed
    best_total = 0
    best_run = 0
    best_offset = None
    n1, n2 = len(seq1), len(seq2)
    for offset in range(-n2 + 1, n1):
        total = 0
        run = 0
        max_run = 0
        for i in range(max(0, offset), min(n1, offset + n2)):
            j = n2 - 1 - (i - offset)  # reverse index into seq2's complement
            if seq1[i] == seq2_rc_source[j]:
                total += 1
                run += 1
                max_run = max(max_run, run)
            else:
                run = 0
        if total > best_total or (total == best_total and max_run > best_run):
            best_total, best_run, best_offset = total, max_run, offset
    return best_total, best_run, best_offset


print("\n=== Primer-primer complementarity check (515F vs 806R) ===")
total_bp, longest_run, _ = best_complementary_alignment(FWD_PRIMER, REV_PRIMER)
three_prime_fwd = FWD_PRIMER[-5:]
three_prime_rev = REV_PRIMER[-5:]
_, run_3p_fwd, _ = best_complementary_alignment(three_prime_fwd, REV_PRIMER)
_, run_3p_rev, _ = best_complementary_alignment(FWD_PRIMER, three_prime_rev)

print(f"Best overall alignment: {total_bp} complementary bp, longest run {longest_run} bp")
print(f"Longest complementary run involving either primer's last 5 bases (3' end): "
      f"{max(run_3p_fwd, run_3p_rev)} bp")

risk = (longest_run >= 4) or (max(run_3p_fwd, run_3p_rev) >= 4) or (total_bp >= 8)
verdict = "FLAG -- re-check with a thermodynamic tool (primer3, IDT OligoAnalyzer, etc.)" if risk else "PASS -- low dimer risk by this heuristic"
print(f"Verdict: {verdict}")
print("(This is a conservative sequence-based heuristic, not a dG/Tm prediction. For a second "
      "opinion, paste both primers into IDT's OligoAnalyzer (free, web-based, no install needed) "
      "or install primer3-py in an environment pinned to Python <=3.13.)")

=== synDNA_GC26_515F806R ===
target GC: 26.0%  actual GC: 26.45%  constraints pass: True
515F at 201-219 correct: True  |  806R at 434-453 correct: True
===> SUCCESS - all constraints evaluations pass
✔PASS ┍ EnforceSequence[200-219]
      │ Enforced by nucleotides restrictions
✔PASS ┍ EnforceSequence[433-453]
      │ Enforced by nucleotides restrictions
✔PASS ┍ EnforceGCContent[0-2000](mini:0.26, maxi:0.27)
      │ Passed !
✔PASS ┍ AvoidHairpins[0-2000](stem_size:15, hairpin_window:200)
      │ Score:         0. Locations: []
✔PASS ┍ UniquifyAllKmers[0-2000(+)](k:15)
      │ Passed: no nonunique 15-mer found.



=== synDNA_GC36_515F806R ===
target GC: 36.0%  actual GC: 35.50%  constraints pass: True
515F at 201-219 correct: True  |  806R at 434-453 correct: True
===> SUCCESS - all constraints evaluations pass
✔PASS ┍ EnforceSequence[200-219]
      │ Enforced by nucleotides restrictions
✔PASS ┍ EnforceSequence[433-453]
      │ Enforced by nucleotides restrictions
✔PASS ┍ EnforceGCConte